# Twitch -> Silver y DLQ

Normaliza el ranking mensual y conserva una fila por mes y posición. El matching con Steam queda para Gold y no genera rechazos.

In [ ]:
import json
dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("silver_schema", "silver")
dbutils.widgets.text("dlq_schema", "dlq")
dbutils.widgets.text("cutoff_us", "")
dbutils.widgets.text("run_id", "")
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
silver_schema = dbutils.widgets.get("silver_schema")
dlq_schema = dbutils.widgets.get("dlq_schema")
cutoff_us = dbutils.widgets.get("cutoff_us")
run_id = dbutils.widgets.get("run_id")

In [ ]:
%sql
-- Silver: actividad Twitch válida para 2016-2024.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :silver_schema || '.twitch_activity_monthly') AS
WITH typed AS (
  SELECT try_cast(Rank AS INT) AS twitch_rank, trim(Game) AS game,
    lower(trim(Game)) AS game_normalized, try_cast(Month AS INT) AS month,
    try_cast(Year AS INT) AS year, make_date(try_cast(Year AS INT), try_cast(Month AS INT), 1) AS month_start,
    try_cast(Hours_watched AS BIGINT) AS hours_watched,
    try_cast(Hours_streamed AS BIGINT) AS hours_streamed,
    try_cast(Peak_viewers AS BIGINT) AS peak_viewers,
    try_cast(Peak_channels AS BIGINT) AS peak_channels,
    try_cast(Streamers AS BIGINT) AS streamers,
    try_cast(Avg_viewers AS BIGINT) AS avg_viewers,
    try_cast(Avg_channels AS BIGINT) AS avg_channels,
    try_cast(Avg_viewer_ratio AS DECIMAL(18,4)) AS avg_viewer_ratio,
    _ingested_at, _source
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.twitch_top_200_games_data')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), ranked AS (
  SELECT *, row_number() OVER (PARTITION BY month_start, twitch_rank ORDER BY hours_watched DESC NULLS LAST, game_normalized ASC NULLS LAST, _ingested_at DESC) AS rn
  FROM typed
), checked AS (
  SELECT *, filter(array(
    CASE WHEN month_start IS NULL OR year < 2016 OR year > 2024 THEN 'fecha_invalida' END,
    CASE WHEN twitch_rank IS NULL OR twitch_rank < 1 OR twitch_rank > 200 THEN 'rank_fuera_de_rango' END,
    CASE WHEN game IS NULL OR game = '' THEN 'game_vacio' END,
    CASE WHEN hours_watched < 0 OR hours_streamed < 0 OR peak_viewers < 0 OR peak_channels < 0 OR streamers < 0 OR avg_viewers < 0 OR avg_channels < 0 OR avg_viewer_ratio < 0 THEN 'metrica_negativa' END
  ), x -> x IS NOT NULL) AS error_reasons
  FROM ranked WHERE rn = 1
)
SELECT twitch_rank, game, game_normalized, month, year, month_start,
  hours_watched, hours_streamed, peak_viewers, peak_channels, streamers,
  avg_viewers, avg_channels, avg_viewer_ratio, _ingested_at, _source
FROM checked WHERE size(error_reasons) = 0;

In [ ]:
%sql
-- DLQ: conserva métricas Twitch inválidas y su payload original tipado.
CREATE OR REPLACE TABLE IDENTIFIER(:catalog || '.' || :dlq_schema || '.twitch_activity_monthly_rejected') AS
WITH typed AS (
  SELECT *, try_cast(Rank AS INT) AS twitch_rank_typed, try_cast(Month AS INT) AS month_typed, try_cast(Year AS INT) AS year_typed,
    make_date(try_cast(Year AS INT), try_cast(Month AS INT), 1) AS month_start,
    try_cast(Hours_watched AS BIGINT) AS hours_watched_typed, try_cast(Hours_streamed AS BIGINT) AS hours_streamed_typed,
    try_cast(Peak_viewers AS BIGINT) AS peak_viewers_typed, try_cast(Peak_channels AS BIGINT) AS peak_channels_typed,
    try_cast(Streamers AS BIGINT) AS streamers_typed, try_cast(Avg_viewers AS BIGINT) AS avg_viewers_typed,
    try_cast(Avg_channels AS BIGINT) AS avg_channels_typed, try_cast(Avg_viewer_ratio AS DECIMAL(18,4)) AS avg_viewer_ratio_typed
  FROM IDENTIFIER(:catalog || '.' || :bronze_schema || '.twitch_top_200_games_data')
  WHERE _ingested_at <= timestamp_micros(try_cast(:cutoff_us AS BIGINT))
), checked AS (
  SELECT *, filter(array(
    CASE WHEN month_start IS NULL OR year_typed < 2016 OR year_typed > 2024 THEN 'fecha_invalida' END,
    CASE WHEN twitch_rank_typed IS NULL OR twitch_rank_typed < 1 OR twitch_rank_typed > 200 THEN 'rank_fuera_de_rango' END,
    CASE WHEN Game IS NULL OR trim(Game) = '' THEN 'game_vacio' END,
    CASE WHEN hours_watched_typed < 0 OR hours_streamed_typed < 0 OR peak_viewers_typed < 0 OR peak_channels_typed < 0 OR streamers_typed < 0 OR avg_viewers_typed < 0 OR avg_channels_typed < 0 OR avg_viewer_ratio_typed < 0 THEN 'metrica_negativa' END
  ), x -> x IS NOT NULL) AS error_reasons
  FROM typed
)
SELECT sha2(concat_ws('|', coalesce(cast(month_start AS STRING), 'NULL'), coalesce(cast(twitch_rank_typed AS STRING), 'NULL'), to_json(struct(*))), 256) AS _reject_key,
  twitch_rank_typed AS twitch_rank, month_start, array_join(error_reasons, '; ') AS motivo_rechazo,
  error_reasons AS reglas_violadas, to_json(struct(*)) AS payload,
  :run_id AS _run_id, current_timestamp() AS _rejected_at
FROM checked WHERE size(error_reasons) > 0;

In [ ]:
dbutils.notebook.exit('{"status":"completed","table":"twitch_activity_monthly"}')